# v7 paged-KV decode — the gate (vast.ai T4)

Standalone, **Run All** top-to-bottom. v7 = v6 split-KV (Flash-Decoding) carried unchanged plus ONE
new variable: the KV reads **gather through a per-sequence block table** (paged KV — the layout a
from-scratch mini-vLLM consumes) instead of a contiguous slice. Two harness fixes ride along: a
`--batch` sweep (measures the occupancy->bandwidth crossover) and a causal **query-offset** (places
the decode query at `N_k-1` so causal decode attends the whole cache).

v7 is deliberately **occupancy-neutral**: `AI = 2/b = 1.0`, byte-identical to v6 (the gather adds
~0.1% index reads). It does NOT attack the limiter — it sets up v8 (GQA M-packing).

Gate (counter-free — `ncu` is blocked on vast.ai): **(1) correctness vs SDPA** (paged + v6 regression)
+ **(2) decode bench** incl. the **`--batch` crossover sweep**. **Pick a T4 GPU + CUDA-devel image.**


## 0. Dependencies + GPU (venv-safe)
Installs into **this kernel's** Python (works with or without a venv) and prepends the kernel's
`bin/` to PATH so every later `!python -m …` cell resolves. Torch is only installed if missing.


In [ ]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

try:
    import torch  # already on the image? keep it, don't churn the version
except ImportError:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
pip('ninja', 'pytest', 'numpy')

# vast.ai venv fix: !-cells spawn a bare shell without the venv on PATH -> `python` not found.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')

import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap --format=csv
!which python && python -c "import torch; print('shell python sees torch', torch.__version__)"

## 1. Get the repo
Clones if absent, otherwise pulls the latest `main`. Adds the repo root to `sys.path`. Safe to re-run.


In [ ]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

## 2. Roofline first (decode `AI = 2/b`, UNCHANGED from v6)
v7 paging is byte-neutral: work = 4·N_k·d FLOPs, traffic = 2·N_k·d·b bytes (read K and V) ⇒ `AI = 2/b`,
N_k-independent. The block-table gather adds `O(N_k/page_size)` int32 index reads (~0.1% of KV bytes)
— negligible. FP16 KV (`b=2`) ⇒ `AI = 1.0`. Expect every row `HBM`; the wall-clock GAP vs this floor
is the occupancy story the `--batch` sweep measures.


In [ ]:
from roofline.archs import get_arch
from roofline.model import estimate
arch = get_arch('sm_75')
print(f"{'shape q/kv':>14} | {'limiter':>7} | {'AI':>4} | {'ridge':>6} | {'t_hbm floor':>11}")
for d in (64, 128):
    for Nk in (2048, 8192, 16384):
        e = estimate(arch, B=1, H=8, N_q=1, N_k=Nk, d=d, precision='fp16', materialize_s=False)
        print(f"{f'1x8x1x{d}/{Nk}':>14} | {e.limiter.upper():>7} | {e.arithmetic_intensity:4.2f} | "
              f"{e.ridge:6.1f} | {e.t_hbm*1e3:8.4f}ms")

## 3. Build v7 (JIT)
First call compiles with nvcc (~1 min), cached after. The clean-up guard removes any interrupted
build dir (a version-stamped dir with no `.so`) that would otherwise make torch skip the rebuild.


In [ ]:
import glob, os, shutil
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_v7_paged')):
    if not glob.glob(os.path.join(d, '*.so')):
        shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
from bindings.load import build_kernel
mod = build_kernel('v7_paged')
print('built:', mod)

## 4. Correctness gate — v7 paged + v6 regression  *(Gate 1 of 2)*
`-k "v7_paged or v6_splitkv"` runs: `test_v7_paged_decode` (scatter dense KV into **shuffled** physical
pages + block table; `N_q=1`, `N_k ∈ {4096,8192,8190}`, page_size ∈ {128,256}, d 64/128, causal both
ways — causal places q at `N_k-1` so it equals the full scan), `test_v7_paged_square_reduces` (the
`num_splits→1` prefill regression through the paged path), AND v6's cases (must still pass — v6 is
untouched). Tolerance 2e-2 (FP16-in). **Must be all-green.**


In [ ]:
!python -m pytest tests/test_correctness.py -k "v7_paged or v6_splitkv" -q

## 5. Decode benchmark — paged, non-causal + causal
`--decode` (N_q=1, swept N is the KV length). Columns: **µs/tok**, **%HBM**, **vs sdpa**, **vs naive**
(v5 at `N_q=1`). With the query-offset fix the **causal** rows now do full-cache work (≈ the non-causal
latency) instead of the degenerate 1-key short-circuit. Paste rows into `docs/results.md` Step 7.


In [ ]:
!python -m bench.harness --backend v7_paged --decode
print()
!python -m bench.harness --backend v7_paged --decode --causal

## 6. The crown jewel — `--batch` crossover sweep (the v7 deliverable)
Sweep `B ∈ {1,8,16,32,64}` at fixed `N_k=8192` (so `BH = 8 → 512`). This MEASURES the
occupancy→bandwidth crossover the re-plan only predicted: `choose_splits` self-disables splitting once
`BH ≥ 2·SM = 80` (T4), so **%HBM should climb from ~12% (BH=8) toward saturation past BH≈80–160**.
This is the empirical basis for the whole "occupancy before bytes" reorder.

After the run: read off the BH where %HBM stops being occupancy-limited, and update
`docs/diagrams/decode-roofline-crossover.svg` from *predicted* to *measured*.


In [ ]:
!python -m bench.harness --backend v7_paged --decode --seq 8192 --batch-sweep 1 8 16 32 64